<a href="https://colab.research.google.com/github/cobaltsleet49/VibeCheck/blob/main/Copy_of_ML_Group11_LoanPrediction.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# please feel free to make changes as you please
from huggingface_hub import hf_hub_download
from sklearn.linear_model import LogisticRegression
from imblearn.over_sampling import SMOTE
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import (AdaBoostClassifier, GradientBoostingClassifier, RandomForestClassifier, StackingClassifier)
from sklearn.metrics import (accuracy_score, confusion_matrix, ConfusionMatrixDisplay, f1_score, precision_recall_curve, precision_score, recall_score)
from sklearn.preprocessing import (OneHotEncode, LabelEncoder, StandardScaler)
from sklearn.model_selection import (train_test_split, GridSearchCV)

import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", None)


In [ ]:
# this cell takes a while to run watch out
csv_path = hf_hub_download(
    repo_id="beedubb/hdmaLARdata",
    filename="2025_public_lar_csv.csv",
    repo_type="dataset",
)

# 10% sampling
df = pd.read_csv(
    csv_path,
    skiprows=lambda i: i > 0 and np.random.rand() > 0.1, # change this float, 0.1 = 10%, 0.2=20% and so on
    low_memory=False,
)

print(f"Successfully loaded shape: {df.shape}")

Successfully loaded shape: (1354928, 99)


In [ ]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1354928 entries, 0 to 1354927
Data columns (total 99 columns):
 #   Column                                    Non-Null Count    Dtype  
---  ------                                    --------------    -----  
 0   activity_year                             1354928 non-null  int64  
 1   lei                                       1354928 non-null  object 
 2   derived_msa_md                            1354928 non-null  int64  
 3   state_code                                1337779 non-null  object 
 4   county_code                               1324665 non-null  float64
 5   census_tract                              1319109 non-null  object 
 6   conforming_loan_limit                     1349240 non-null  object 
 7   derived_loan_product_type                 1354928 non-null  object 
 8   derived_dwelling_category                 1354928 non-null  object 
 9   derived_ethnicity                         1354928 non-null  object 
 10  derive

In [ ]:
df.head(10)

,activity_year,lei,derived_msa_md,state_code,county_code,census_tract,conforming_loan_limit,derived_loan_product_type,derived_dwelling_category,derived_ethnicity,derived_race,derived_sex,action_taken,purchaser_type,preapproval,loan_type,loan_purpose,lien_status,reverse_mortgage,open_end_line_of_credit,business_or_commercial_purpose,loan_amount,combined_loan_to_value_ratio,interest_rate,rate_spread,hoepa_status,total_loan_costs,total_points_and_fees,origination_charges,discount_points,lender_credits,loan_term,prepayment_penalty_term,intro_rate_period,negative_amortization,interest_only_payment,balloon_payment,other_nonamortizing_features,property_value,construction_method,occupancy_type,manufactured_home_secured_property_type,manufactured_home_land_property_interest,total_units,multifamily_affordable_units,income,debt_to_income_ratio,applicant_credit_score_type,co_applicant_credit_score_type,applicant_ethnicity_1,applicant_ethnicity_2,applicant_ethnicity_3,applicant_ethnicity_4,applicant_ethnicity_5,co_applicant_ethnicity_1,co_applicant_ethnicity_2,co_applicant_ethnicity_3,co_applicant_ethnicity_4,co_applicant_ethnicity_5,applicant_ethnicity_observed,co_applicant_ethnicity_observed,applicant_race_1,applicant_race_2,applicant_race_3,applicant_race_4,applicant_race_5,co_applicant_race_1,co_applicant_race_2,co_applicant_race_3,co_applicant_race_4,co_applicant_race_5,applicant_race_observed,co_applicant_race_observed,applicant_sex,co_applicant_sex,applicant_sex_observed,co_applicant_sex_observed,applicant_age,co_applicant_age,applicant_age_above_62,co_applicant_age_above_62,submission_of_application,initially_payable_to_institution,aus_1,aus_2,aus_3,aus_4,aus_5,denial_reason_1,denial_reason_2,denial_reason_3,denial_reason_4,tract_population,tract_minority_population_percent,ffiec_msa_md_median_family_income,tract_to_msa_income_percentage,tract_owner_occupied_units,tract_one_to_four_family_homes,tract_median_age_of_housing_units
0,2025,549300QFJDBOCX7WKY08,46060,AZ,4019.0,04019004428,C,Conventional:First Lien,Single Family (1-4 Units):Site-Built,Ethnicity Not Available,Race Not Available,Sex Not Available,1,0,2,1,1,1,2,2,1,625000,49.408,7.25,NaN,3,NaN,NaN,NaN,NaN,NaN,60,NaN,NaN,2,2,1,2,1265000,1,2,3,5,1,NaN,NaN,NaN,9,10,4.0,NaN,NaN,NaN,NaN,5.0,NaN,NaN,NaN,NaN,3,4,7.0,NaN,NaN,NaN,NaN,8.0,NaN,NaN,NaN,NaN,3,4,4,5,3,4,8888,9999,NaN,NaN,1,1,6,NaN,NaN,NaN,NaN,10,NaN,NaN,NaN,3187,26.61,96100,180.0,1599,1765,32
1,2025,549300QFJDBOCX7WKY08,23104,TX,48439.0,48439100900,C,Conventional:First Lien,Single Family (1-4 Units):Site-Built,Ethnicity Not Available,Race Not Available,Sex Not Available,1,0,2,1,31,1,2,2,1,435000,52.575,7,NaN,3,NaN,NaN,NaN,NaN,NaN,120,NaN,60,2,2,1,2,395000,1,3,3,5,2,NaN,NaN,NaN,9,10,4.0,NaN,NaN,NaN,NaN,5.0,NaN,NaN,NaN,NaN,3,4,7.0,NaN,NaN,NaN,NaN,8.0,NaN,NaN,NaN,NaN,3,4,4,5,3,4,8888,9999,NaN,NaN,1,1,6,NaN,NaN,NaN,NaN,10,NaN,NaN,NaN,1854,94.88,106900,67.0,230,772,0
2,2025,254900Y3KT6SZM11MQ06,34980,TN,47189.0,47189030102,NC,Conventional:First Lien,Single Family (1-4 Units):Site-Built,Not Hispanic or Latino,White,Male,4,0,2,1,31,1,1111,1111,1111,1415000,Exempt,Exempt,Exempt,3,Exempt,Exempt,Exempt,Exempt,Exempt,Exempt,Exempt,Exempt,1111,1111,1111,1111,Exempt,1,1,1111,1111,1,Exempt,213.0,Exempt,1111,1111,2.0,NaN,NaN,NaN,NaN,5.0,NaN,NaN,NaN,NaN,2,4,5.0,NaN,NaN,NaN,NaN,8.0,NaN,NaN,NaN,NaN,2,4,1,5,2,4,55-64,9999,No,NaN,1111,1111,1111,NaN,NaN,NaN,NaN,1111,NaN,NaN,NaN,7100,14.85,112200,91.0,2327,2808,29
3,2025,254900Y3KT6SZM11MQ06,26420,TX,48201.0,48201250602,C,Conventional:First Lien,Single Family (1-4 Units):Site-Built,Ethnicity Not Available,Race Not Available,Sex Not Available,1,0,2,1,1,1,1111,1111,1111,165000,Exempt,Exempt,Exempt,3,Exempt,Exempt,Exempt,Exempt,Exempt,Exempt,Exempt,Exempt,1111,1111,1111,1111,Exempt,1,3,1111,1111,1,Exempt,NaN,Exempt,1111,1111,4.0,NaN,NaN,NaN,NaN,5.0,NaN,NaN,NaN,NaN,3,4,7.0,NaN,NaN,NaN,NaN,8.0,NaN,NaN,NaN,NaN,3,4,4,5,3,4,8888,9999,NaN,NaN,1111,1111,1111,NaN,NaN,NaN,NaN,1111,NaN,NaN,NaN,4792,77.78,1

In [ ]:
target_column = "action_taken"

# ratio of null values to filled values, higher null percentage means that most of the values are null values
pd.set_option("display.max_rows", None)

null_df = ((df.isnull().sum() / len(df)) * 100).to_frame()
null_df.columns = ["null_percentage"]
temp = null_df[null_df["null_percentage"]>=0].sort_values(by="null_percentage", ascending=False)
display(temp)

pd.reset_option("display.max_rows")

# print("        Null percentages       ")
# print("-------------------------------")
# for col, num in temp.iterrows():
#   print(f" {num['null_percentage']:.4f}% | {col}")


,null_percentage
co_applicant_ethnicity_5,99.999188
co_applicant_ethnicity_4,99.998819
applicant_ethnicity_5,99.998081
applicant_ethnicity_4,99.996310
co_applicant_race_5,99.991365
co_applicant_race_4,99.979187
applicant_race_5,99.977268
co_applicant_ethnicity_3,99.959481
applicant_race_4,99.938373
applicant_ethnicity_3,99.883462


In [ ]:
# checking duplicates
# not too sure how to check this one
numDupes = df.duplicated().sum()
print(f"Number of duplicate entries: {numDupes}")

Number of duplicate entries: 649


In [ ]:
# this was just for copypasting the string names in the next cell
for col, dtype in df.dtypes.items():
  # print(f"{col}: {dtype}")
  print(f"\"{col}\",")

"activity_year",
"lei",
"derived_msa_md",
"state_code",
"county_code",
"census_tract",
"conforming_loan_limit",
"derived_loan_product_type",
"derived_dwelling_category",
"derived_ethnicity",
"derived_race",
"derived_sex",
"action_taken",
"purchaser_type",
"preapproval",
"loan_type",
"loan_purpose",
"lien_status",
"reverse_mortgage",
"open_end_line_of_credit",
"business_or_commercial_purpose",
"loan_amount",
"combined_loan_to_value_ratio",
"interest_rate",
"rate_spread",
"hoepa_status",
"total_loan_costs",
"total_points_and_fees",
"origination_charges",
"discount_points",
"lender_credits",
"loan_term",
"prepayment_penalty_term",
"intro_rate_period",
"negative_amortization",
"interest_only_payment",
"balloon_payment",
"other_nonamortizing_features",
"property_value",
"construction_method",
"occupancy_type",
"manufactured_home_secured_property_type",
"manufactured_home_land_property_interest",
"total_units",
"multifamily_affordable_units",
"income",
"debt_to_income_ratio",
"applicant_cred

In [ ]:
# Data preprocessing

# removing unnecessary features, add more or remove if need be
features_to_drop = [
    "lei",
    "activity_year",
    "applicant_ethnicity_1",
    "applicant_ethnicity_2",
    "applicant_ethnicity_3",
    "applicant_ethnicity_4",
    "applicant_ethnicity_5",
    "co_applicant_ethnicity_1",
    "co_applicant_ethnicity_2",
    "co_applicant_ethnicity_3",
    "co_applicant_ethnicity_4",
    "co_applicant_ethnicity_5",
    "applicant_ethnicity_observed",
    "co_applicant_ethnicity_observed",
    "applicant_race_1",
    "applicant_race_2",
    "applicant_race_3",
    "applicant_race_4",
    "applicant_race_5",
    "co_applicant_race_1",
    "co_applicant_race_2",
    "co_applicant_race_3",
    "co_applicant_race_4",
    "co_applicant_race_5",
    "aus_1",
    "aus_2",
    "aus_3",
    "aus_4",
    "aus_5",
    "denial_reason_1",
    "denial_reason_2",
    "denial_reason_3",
]

# df.drop(columns=features_to_drop)

# Drop or fill in null values with median/averages

# Perform feature selection (filter, wrapper, etc.) if still too many features

# Scale features

# Apply SMOTE because dataset is heavily biased towards loan approval

